# DualSentinel — Run Pipeline (LMD-2023, full evaluation)

Executa o pipeline completo (preprocessor + detectors + SLM Analyst + LLM Judge) sobre
`LMD-2023 [1.75M Elements - Normal]checked.csv`, com `--evaluate` e `--seed 42`.

**Sem `--max-llm-calls`** → todas as janelas acima do threshold passam pelo Ollama. Pode demorar bastante.

Comando equivalente:

```bash
python src/pipeline.py --input "data/samples/LMD-2023 [1.75M Elements - Normal]checked.csv" --dataset lmd --evaluate --seed 42
```


In [ ]:
import os, sys, json, subprocess
from pathlib import Path
import pandas as pd
from IPython.display import Markdown, display

NB_DIR     = Path.cwd()
DS_DIR     = NB_DIR if (NB_DIR / "src" / "pipeline.py").exists() else NB_DIR.parent
REPO_ROOT  = DS_DIR.parent
INPUT_REL  = "data/samples/LMD-2023 [1.75M Elements - Normal]checked.csv"
INPUT_CSV  = DS_DIR / INPUT_REL
RESULTS_DIR = DS_DIR / "results"

assert (DS_DIR / "src" / "pipeline.py").exists(), f"src/pipeline.py não encontrado em {DS_DIR}"
assert INPUT_CSV.exists(), f"Input não encontrado: {INPUT_CSV}"

print("DS_DIR    :", DS_DIR)
print("INPUT_CSV :", INPUT_CSV, f"({INPUT_CSV.stat().st_size / 1e6:.1f} MB)")
print("RESULTS   :", RESULTS_DIR)


## Verify Ollama service

Confirma que o daemon do Ollama responde em `localhost:11434` e que os modelos default
(`SLM_MODEL` e `JUDGE_MODEL`, ler do `.env` se existir) estão disponíveis.


In [ ]:
import urllib.request

OLLAMA_URL = "http://localhost:11434/api/tags"
SLM_MODEL   = os.getenv("SLM_MODEL", "phi3:mini")
JUDGE_MODEL = os.getenv("JUDGE_MODEL", "llama3.2")

try:
    with urllib.request.urlopen(OLLAMA_URL, timeout=3) as r:
        tags = json.loads(r.read().decode())
    installed = sorted({m["name"] for m in tags.get("models", [])})
    print("Ollama OK — modelos instalados:")
    for n in installed:
        print(" -", n)
    for needed in (SLM_MODEL, JUDGE_MODEL):
        present = any(n.startswith(needed.split(":")[0]) for n in installed)
        flag = "OK" if present else "MISSING"
        print(f"[{flag}] {needed}")
except Exception as e:
    print("Ollama indisponível:", e)
    print("Inicia com: ollama serve")


## Configure pipeline parameters


In [ ]:
DATASET  = "lmd"
SEED     = 42
EVALUATE = True

cmd = [
    sys.executable, "-u", "src/pipeline.py",
    "--input", INPUT_REL,
    "--dataset", DATASET,
    "--seed", str(SEED),
]
if EVALUATE:
    cmd.append("--evaluate")

print("CWD :", DS_DIR)
print("CMD :", " ".join(repr(x) if " " in x else x for x in cmd))


## Execute the pipeline

Streams stdout/stderr line-by-line. Long-running (sem `--max-llm-calls`).


In [ ]:
proc = subprocess.Popen(
    cmd,
    cwd=str(DS_DIR),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env={**os.environ, "PYTHONUNBUFFERED": "1"},
)
try:
    for line in proc.stdout:
        print(line, end="")
finally:
    rc = proc.wait()

print(f"\n[exit code: {rc}]")
assert rc == 0, f"Pipeline falhou com exit code {rc}"


## Locate latest results directory


In [ ]:
import re
RUN_RE = re.compile(r"^\d{4}-\d{2}-\d{2}_\d{2}-\d{2}$")

run_dirs = sorted(
    [p for p in RESULTS_DIR.iterdir() if p.is_dir() and RUN_RE.match(p.name)],
    key=lambda p: p.stat().st_mtime,
)
assert run_dirs, f"Sem run dirs em {RESULTS_DIR}"
RUN_DIR = run_dirs[-1]

print("RUN_DIR:", RUN_DIR)
for f in sorted(RUN_DIR.iterdir()):
    print(f"  {f.name:<40} {f.stat().st_size/1024:>10.1f} KB")


## Scored windows


In [ ]:
windows = json.loads((RUN_DIR / "windows_scored.json").read_text(encoding="utf-8"))
df_w = pd.DataFrame(windows)
print(f"Total janelas: {len(df_w)}")
display(df_w["detector_score"].describe().to_frame("detector_score"))

cols = [c for c in ("window_start", "window_end", "event_count", "detector_score") if c in df_w.columns]
display(df_w.sort_values("detector_score", ascending=False)[cols].head(15))


## SLM Analyst pre-diagnoses (Phi-3)


In [ ]:
slm_path = RUN_DIR / "slm_analyses.json"
if slm_path.exists():
    slm = json.loads(slm_path.read_text(encoding="utf-8"))
    df_s = pd.DataFrame(slm)
    print(f"SLM analyses: {len(df_s)}")
    cols = [c for c in ("window_start", "pre_score", "needs_deep_analysis",
                        "suspected_techniques", "indicators", "summary") if c in df_s.columns]
    display(df_s[cols].head(20))
else:
    print("slm_analyses.json não existe (provavelmente --skip-judge).")


## LLM Judge verdicts (Llama)


In [ ]:
judge_path = RUN_DIR / "judge_results.json"
if judge_path.exists():
    judges = json.loads(judge_path.read_text(encoding="utf-8"))
    df_j = pd.DataFrame(judges)
    print(f"Judge results: {len(df_j)}")
    print("Verdict distribution:")
    if "verdict" in df_j.columns:
        display(df_j["verdict"].value_counts().to_frame("count"))
    cols = [c for c in ("window_start", "anomaly_score", "verdict", "fp_risk",
                        "techniques", "rationale", "recommended_action",
                        "model_used", "latency_s") if c in df_j.columns]
    display(df_j.sort_values("anomaly_score", ascending=False)[cols].head(20))
else:
    print("judge_results.json não existe.")


## Evaluation metrics


In [ ]:
metrics_path = RUN_DIR / "metrics.json"
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text(encoding="utf-8"))

    headline_keys = ("precision", "recall", "f1", "roc_auc", "pr_auc",
                     "accuracy", "threshold", "n_windows", "n_positive")
    headline = {k: metrics[k] for k in headline_keys if k in metrics}
    if headline:
        display(pd.Series(headline, name="value").to_frame())

    cm = metrics.get("confusion_matrix")
    if cm:
        display(pd.DataFrame(cm,
            index=["actual_neg", "actual_pos"],
            columns=["pred_neg", "pred_pos"]))

    tx = metrics.get("techniques") or metrics.get("multi_label") or {}
    if tx:
        df_tx = pd.DataFrame(tx).T if isinstance(tx, dict) else pd.DataFrame(tx)
        print("\nMITRE multi-label metrics:")
        display(df_tx.head(30))
else:
    print("metrics.json não existe — correr com --evaluate.")


## Run manifest and telemetry


In [ ]:
manifest_path = RUN_DIR / "run_manifest.json"
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    print("=== run_manifest.json ===")
    for k in ("input_sha256", "models", "seed", "threshold", "git_commit",
              "platform", "started_at", "finished_at", "duration_s"):
        if k in manifest:
            print(f"  {k:<14}: {manifest[k]}")
else:
    print("run_manifest.json não existe.")

telemetry_path = RUN_DIR / "telemetry.json"
if telemetry_path.exists():
    telemetry = json.loads(telemetry_path.read_text(encoding="utf-8"))
    print("\n=== telemetry.json (aggregated) ===")
    if isinstance(telemetry, dict) and "summary" in telemetry:
        display(pd.DataFrame(telemetry["summary"]).T)
    elif isinstance(telemetry, list):
        df_t = pd.DataFrame(telemetry)
        if "stage" in df_t.columns:
            agg = df_t.groupby("stage").agg(
                calls=("ok", "count"),
                ok=("ok", "sum"),
                total_s=("duration_s", "sum"),
                mean_s=("duration_s", "mean"),
                prompt_tokens=("prompt_tokens", "sum"),
                completion_tokens=("completion_tokens", "sum"),
            )
            display(agg)
        else:
            display(df_t.head())
else:
    print("telemetry.json não existe.")


## Markdown report


In [ ]:
reports = sorted(RUN_DIR.glob("report_*.md"))
if reports:
    md = reports[-1].read_text(encoding="utf-8")
    print(f"Render: {reports[-1].name}\n")
    display(Markdown(md))
else:
    print("Nenhum report_*.md encontrado.")
